In [ ]:
#!/usr/bin/env python3
"""
Calculate the two torsional angles of Thiazole Orange.

φ1: S-C4-C6-C2
φ2: C2-C6-C4-N1

Angles from individual replicas are stored separately.

Example:
    python calculate_to_torsions.py \
        --manifest configs/systems/torsion_na_antiparallel.local.csv \
        --output data/processed/torsion/torsion_na_antiparallel.npz \
        --stride 10
"""

import argparse
import csv
from collections import OrderedDict
from pathlib import Path

import MDAnalysis as mda
from MDAnalysis.lib.distances import calc_dihedrals
import numpy as np


TORSION_ATOMS = {
    "S": "resname TOG and name S",
    "C4": "resname TOG and name C4",
    "C6": "resname TOG and name C6",
    "C2": "resname TOG and name C2",
    "N1": "resname TOG and name N1",
}


def parse_arguments():
    """讀取命令列參數。"""

    parser = argparse.ArgumentParser(
        description="Calculate TO φ1 and φ2 torsional angles."
    )

    parser.add_argument(
        "--manifest",
        required=True,
        type=Path,
        help="CSV file containing topology and trajectory paths.",
    )

    parser.add_argument(
        "--output",
        required=True,
        type=Path,
        help="Output NPZ file.",
    )

    parser.add_argument(
        "--stride",
        type=int,
        default=10,
        help="Analyze every Nth trajectory frame. Default: 10.",
    )

    parser.add_argument(
        "--time-per-frame-ns",
        type=float,
        default=0.002,
        help="Time represented by one original DCD frame. Default: 0.002 ns.",
    )

    return parser.parse_args()


def resolve_path(path_text, manifest_directory):
    """解析 manifest 中的絕對或相對路徑。"""

    path = Path(path_text.strip())

    if path.is_absolute():
        return path

    return manifest_directory / path


def load_manifest(manifest_path):
    """讀取 CSV 並依照 system_id 整理系統資訊。"""

    if not manifest_path.exists():
        raise FileNotFoundError(
            f"找不到 manifest：{manifest_path}"
        )

    required_columns = {
        "system_id",
        "label",
        "color",
        "topology",
        "replica",
        "trajectory",
    }

    systems = OrderedDict()
    manifest_directory = manifest_path.parent

    with manifest_path.open(
        mode="r",
        encoding="utf-8-sig",
        newline="",
    ) as handle:
        reader = csv.DictReader(handle)

        if reader.fieldnames is None:
            raise ValueError("Manifest 沒有標題列。")

        missing_columns = required_columns - set(reader.fieldnames)

        if missing_columns:
            raise ValueError(
                "Manifest 缺少欄位："
                + ", ".join(sorted(missing_columns))
            )

        for row in reader:
            system_id = row["system_id"].strip()

            if not system_id:
                continue

            topology = resolve_path(
                row["topology"],
                manifest_directory,
            )

            trajectory = resolve_path(
                row["trajectory"],
                manifest_directory,
            )

            systems.setdefault(
                system_id,
                {
                    "label": row["label"].strip(),
                    "color": row["color"].strip(),
                    "topology": topology,
                    "replicas": [],
                },
            )

            if systems[system_id]["topology"] != topology:
                raise ValueError(
                    f"{system_id} 使用了不一致的 topology。"
                )

            systems[system_id]["replicas"].append(
                {
                    "replica": int(row["replica"]),
                    "trajectory": trajectory,
                }
            )

    return systems


def select_single_atom(universe, selection, atom_name):
    """執行原子選擇，並確認剛好選到一個原子。"""

    atom_group = universe.select_atoms(selection)

    if len(atom_group) != 1:
        raise ValueError(
            f"{atom_name} 應選到一個原子，"
            f"實際選到 {len(atom_group)} 個：{selection}"
        )

    return atom_group


def calculate_replica_torsions(
    topology_path,
    trajectory_path,
    stride,
    time_per_frame_ns,
):
    """計算單一 replica 的 φ1、φ2 與時間軸。"""

    if not topology_path.exists():
        raise FileNotFoundError(
            f"找不到 topology：{topology_path}"
        )

    if not trajectory_path.exists():
        raise FileNotFoundError(
            f"找不到 trajectory：{trajectory_path}"
        )

    universe = mda.Universe(
        str(topology_path),
        str(trajectory_path),
    )

    atoms = {
        name: select_single_atom(
            universe,
            selection,
            name,
        )
        for name, selection in TORSION_ATOMS.items()
    }

    phi1_values = []
    phi2_values = []
    time_values = []

    for timestep in universe.trajectory[::stride]:
        box = (
            timestep.dimensions
            if timestep.dimensions is not None
            else None
        )

        phi1 = calc_dihedrals(
            atoms["S"].positions,
            atoms["C4"].positions,
            atoms["C6"].positions,
            atoms["C2"].positions,
            box=box,
        )[0]

        phi2 = calc_dihedrals(
            atoms["C2"].positions,
            atoms["C6"].positions,
            atoms["C4"].positions,
            atoms["N1"].positions,
            box=box,
        )[0]

        phi1_values.append(
            np.rad2deg(phi1)
        )

        phi2_values.append(
            np.rad2deg(phi2)
        )

        time_values.append(
            timestep.frame * time_per_frame_ns
        )

    if not phi1_values:
        raise ValueError(
            "Trajectory 中沒有可分析的 frame。"
        )

    return {
        "time": np.asarray(
            time_values,
            dtype=float,
        ),
        "phi1": np.asarray(
            phi1_values,
            dtype=float,
        ),
        "phi2": np.asarray(
            phi2_values,
            dtype=float,
        ),
    }


def main():
    """主程式。"""

    args = parse_arguments()

    if args.stride < 1:
        raise ValueError("--stride 必須至少為 1。")

    systems = load_manifest(
        args.manifest
    )

    save_data = {}
    system_count = 0

    for system_index, (system_id, system_info) in enumerate(
        systems.items()
    ):
        prefix = f"sys_{system_index}"

        print(f"\nProcessing: {system_info['label']}")

        valid_replica_ids = []

        replicas = sorted(
            system_info["replicas"],
            key=lambda item: item["replica"],
        )

        for replica_info in replicas:
            replica_id = replica_info["replica"]
            trajectory = replica_info["trajectory"]

            try:
                result = calculate_replica_torsions(
                    topology_path=system_info["topology"],
                    trajectory_path=trajectory,
                    stride=args.stride,
                    time_per_frame_ns=args.time_per_frame_ns,
                )

            except (OSError, ValueError) as error:
                print(
                    f"  [警告] Replica {replica_id} 失敗：{error}"
                )
                continue

            replica_prefix = (
                f"{prefix}_rep{replica_id}"
            )

            save_data[
                f"{replica_prefix}_time"
            ] = result["time"]

            save_data[
                f"{replica_prefix}_phi1"
            ] = result["phi1"]

            save_data[
                f"{replica_prefix}_phi2"
            ] = result["phi2"]

            valid_replica_ids.append(
                replica_id
            )

            print(
                f"  Replica {replica_id}: "
                f"{len(result['phi1'])} sampled frames"
            )

        if not valid_replica_ids:
            print(
                f"  [跳過] {system_id} 沒有有效 replica。"
            )
            continue

        save_data[
            f"{prefix}_system_id"
        ] = np.asarray(system_id)

        save_data[
            f"{prefix}_label"
        ] = np.asarray(system_info["label"])

        save_data[
            f"{prefix}_color"
        ] = np.asarray(system_info["color"])

        save_data[
            f"{prefix}_replica_ids"
        ] = np.asarray(
            valid_replica_ids,
            dtype=int,
        )

        system_count += 1

    if system_count == 0:
        raise RuntimeError(
            "沒有任何有效 torsion 資料可以儲存。"
        )

    save_data["stride"] = np.asarray(
        args.stride,
        dtype=int,
    )

    save_data["time_per_frame_ns"] = np.asarray(
        args.time_per_frame_ns,
        dtype=float,
    )

    save_data["phi1_definition"] = np.asarray(
        "S-C4-C6-C2"
    )

    save_data["phi2_definition"] = np.asarray(
        "C2-C6-C4-N1"
    )

    args.output.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    np.savez_compressed(
        args.output,
        **save_data,
    )

    print(f"\nTorsion data saved to: {args.output}")


if __name__ == "__main__":
    main()